# Getting Started: OpenAI Agents in an Azure Sandbox

Give an OpenAI agent its own isolated Linux computer in Azure. We wrap
sandbox `exec`, `write_file`, and `read_file` as `@function_tool`s, hand
them to an `Agent`, and ask the agent to introspect its sandbox.

## Prerequisites
- Azure CLI [signed in](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively)
- `azure-containerapps-sandbox` SDK (from GitHub Releases - see [lab README](./README.md#prerequisites)):
  ```bash
  gh release download v0.1.0b1 \
      --repo Azure-Samples/azure-container-apps-sandboxes \
      --pattern "azure_containerapps_sandbox-*.whl" \
      --output azure_containerapps_sandbox.whl
  pip install azure_containerapps_sandbox.whl
  pip install openai-agents
  ```
- An OpenAI provider, configured **once** for the whole lab in `provider_config.py`
  (see the [lab README](./README.md#prerequisites)). Azure OpenAI is preferred and
  uses Entra ID auth by default.

Click **Run All** or go **Step by Step**.

### 0. Initialize variables

In [ ]:
import os, json, shutil, subprocess
from azure.containerapps.sandbox import SandboxClient, SandboxGroupClient

# Derive names from lab folder
lab_name = os.path.basename(os.path.dirname(os.path.abspath(
    globals().get('__vsc_ipynb_file__', os.path.join(os.getcwd(), '__file__')))))

# Resolve full path to the az CLI (on Windows it's az.cmd, which subprocess.run
# can't find by bare name without shell=True).
az = shutil.which('az')
if not az:
    raise RuntimeError(
        'Azure CLI not found on PATH. Install it from '
        'https://learn.microsoft.com/cli/azure/install-azure-cli and re-run.'
    )

account = json.loads(subprocess.run(
    [az, 'account', 'show', '-o', 'json'],
    capture_output=True, text=True, check=True).stdout)

subscription_id = account['id']
resource_group_name = f'lab-{lab_name}'
resource_group_location = 'westus2'
sandbox_group_name = f'{lab_name}-sg'

print(f'Lab:            {lab_name}')
print(f'User:           {account["user"]["name"]}')
print(f'Subscription:   {account["name"]} ({subscription_id})')
print(f'Resource Group: {resource_group_name}')
print(f'Location:       {resource_group_location}')
print(f'Sandbox Group:  {sandbox_group_name}')

client = SandboxClient(subscription_id=subscription_id, resource_group=resource_group_name)
mgmt = SandboxGroupClient(subscription_id=subscription_id, resource_group=resource_group_name)

### 0a. Configure the OpenAI provider

Provider config is shared across the three notebooks in this lab. Copy [`provider_config.py.example`](./provider_config.py.example) to `provider_config.py` (gitignored) and fill in either the Azure OpenAI or OpenAI section -- all three notebooks pick it up automatically. If you prefer environment variables, leave it as-is and set `AZURE_OPENAI_ENDPOINT`/`AZURE_OPENAI_DEPLOYMENT` (Entra ID auth by default) or `OPENAI_API_KEY` in your shell before launching VS Code.

In [ ]:
import sys, os
_lab_dir = os.path.dirname(os.path.abspath(
    globals().get('__vsc_ipynb_file__', os.path.join(os.getcwd(), '__file__'))))
if _lab_dir not in sys.path:
    sys.path.insert(0, _lab_dir)

from provider import get_model
model = get_model()


### 1. Create resource group, sandbox group, and sandbox

In [ ]:
!az group create --name {resource_group_name} --location {resource_group_location} -o none

group = mgmt.create_group(sandbox_group_name, location=resource_group_location)
print(f'Sandbox group: {group.name} ({group.location})')

# Ensure the signed-in user has data-plane access on the sandbox group.
# `mgmt.create_group` only needs ARM (control plane) rights, but
# `client.create_sandbox` calls the data plane (management.azuredevcompute.io)
# which requires a separate role on the sandbox group resource itself.
sg_id = group.id
data_role = 'Dev Compute SandboxGroup Data Owner'

signed_in_oid = subprocess.run(
    [az, 'ad', 'signed-in-user', 'show', '--query', 'id', '-o', 'tsv'],
    capture_output=True, text=True, check=True).stdout.strip()

existing = json.loads(subprocess.run(
    [az, 'role', 'assignment', 'list',
     '--assignee', signed_in_oid,
     '--scope', sg_id,
     '--role', data_role,
     '-o', 'json'],
    capture_output=True, text=True, check=True).stdout)

if not existing:
    print(f'Assigning "{data_role}" to {signed_in_oid} on sandbox group...')
    subprocess.run(
        [az, 'role', 'assignment', 'create',
         '--assignee-object-id', signed_in_oid,
         '--assignee-principal-type', 'User',
         '--role', data_role,
         '--scope', sg_id,
         '-o', 'none'],
        check=True)
    # Role assignments take a few seconds to propagate to the data plane.
    import time
    for i in range(30):
        time.sleep(5)
        try:
            sbx = client.create_sandbox(sandbox_group_name, disk='ubuntu')
            break
        except Exception as e:
            if '403' not in str(e) or i == 29:
                raise
            print(f'  waiting for role propagation... ({(i+1)*5}s)')
    else:
        raise RuntimeError('Role assignment did not propagate within 150s')
else:
    print(f'Role "{data_role}" already assigned.')
    sbx = client.create_sandbox(sandbox_group_name, disk='ubuntu')

sandbox_id = sbx.id
print(f'Sandbox: {sandbox_id}  state={sbx.state}')

### 2. Wrap the SDK as agent tools

We turn three sandbox SDK calls into `@function_tool` callables. The agent picks the tool, supplies the arguments, and we forward the call to the live sandbox.

In [ ]:
from agents import Agent, Runner, function_tool, set_tracing_disabled

# Disable agents tracing. The SDK exports traces to api.openai.com, which
# may be unreachable on networks where only Azure OpenAI is allowed; that
# failure surfaces as a ConnectError masking the successful AOAI call.
set_tracing_disabled(True)

@function_tool
def shell(command: str) -> str:
    """Run a shell command inside the sandbox.

    Returns the exit code, stdout, and stderr concatenated into one string.
    """
    r = client.exec(sandbox_id, sandbox_group_name, command)
    return (
        f'exit_code={r.exit_code}\n'
        f'stdout:\n{r.stdout}\n'
        f'stderr:\n{r.stderr}'
    )

@function_tool
def write_file(path: str, content: str) -> str:
    """Write text content to an absolute path in the sandbox."""
    client.write_file(sandbox_id, sandbox_group_name, path, content)
    return f'wrote {len(content)} bytes to {path}'

@function_tool
def read_file(path: str) -> str:
    """Read a UTF-8 text file from the sandbox."""
    data = client.read_file(sandbox_id, sandbox_group_name, path)
    return data.decode('utf-8', errors='replace')

agent = Agent(
    name='Sandbox Engineer',
    model=model,
    instructions=(
        'You have full access to a Linux sandbox through three tools: '
        '`shell` runs commands, `write_file` writes text files, '
        '`read_file` reads text files. Use them to answer the user.'
    ),
    tools=[shell, write_file, read_file],
)
print('Agent ready with 3 tools:', [t.name for t in agent.tools])

### 3. Run the agent

Ask the agent to introspect the sandbox. Watch it call the tools you just defined.

In [ ]:
question = (
    'Do these three things in order, then give me a one-paragraph summary:\n'
    '1. Run `uname -a` and `whoami` to identify the sandbox.\n'
    '2. Write a file at /tmp/agent-hello.txt containing "Hello from the agent!".\n'
    '3. Read /tmp/agent-hello.txt back and confirm the contents match what you wrote.'
)

result = await Runner.run(agent, question)
print(result.final_output)

### 4. Clean up

In [ ]:
client.delete_sandbox(sandbox_id, sandbox_group_name)
print(f'Deleted sandbox: {sandbox_id}')

mgmt.delete_group(sandbox_group_name)
print(f'Deleted group: {sandbox_group_name}')

!az group delete --name {resource_group_name} --yes --no-wait
print(f'Deleting resource group: {resource_group_name}')